##import

In [1]:
import pandas as pd
import json
from pathlib import Path
import numpy as np

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score

###data


In [5]:
df_department = pd.read_csv("/content/department-v2.csv")
df_seniority = pd.read_csv("/content/seniority-v2.csv")

In [6]:
def workers_positions_to_csv(data):
    people = json.loads(Path(data).read_text(encoding="utf-8"))
    rows = []

    for person_idx, jobs in enumerate(people, start=1):
        jobs_sorted = sorted(jobs, key=lambda d: (d.get("startDate") is None, d.get("startDate")))
        for job_idx, j in enumerate(jobs_sorted, start=1):
            rows.append({
                "person_id": person_idx,
                "job_no": job_idx,
                "organization": j.get("organization"),
                "linkedin": j.get("linkedin"),
                "position": j.get("position"),
                "startDate": j.get("startDate"),
                "endDate": j.get("endDate"),
                "status": j.get("status"),
                "department": j.get("department"),
                "seniority": j.get("seniority"),
            })

    df = pd.DataFrame(rows, columns=[
        "person_id","job_no","organization","linkedin","position",
        "startDate","endDate","status","department","seniority"
    ])
    df["person_id"] = df["person_id"].astype(int)
    df["job_no"] = df["job_no"].astype(int)
    return df

In [7]:
df_t_annotated= workers_positions_to_csv("/content/linkedin-cvs-annotated.json")
df_t_notannotated= workers_positions_to_csv("/content/linkedin-cvs-not-annotated.json")

In [8]:
df_pred_bow = pd.read_csv("/content/df_t_notannotated_pred_bow_both_threshold.csv")
df_pred_emb = pd.read_csv("/content/df_t_notannotated_pred_embeddings_both_threshold.csv")

#Approach

###Embedding and Bag-of-Words models on extended LinkedIn data

I trained and evaluated both embedding-based and Bag-of-Words (TF–IDF) Logistic Regression models on an extended version of the v2 datasets, created by augmenting the original label-list data with predictions from unannotated LinkedIn job titles.

To control class imbalance and prevent very large labels from dominating training, I applied label-wise downsampling before adding the new data. Each label was capped at a maximum number of examples (500, 400, or 300).

The motivation for this was that, in earlier Bag-of-Words, training on a reduced and more balanced label distribution led to more stable decision boundaries and improved generalization, especially for minority labels. Based on this observation, I reused the idea here but with higher caps than in earlier downsampling experiments.

For the same reason, I also experimented again with higher-than-usual train/test split ratios.

##Embedding-based classification
Job titles were embedded using a pretrained multilingual sentence **embedding model:**
- paraphrase-multilingual-mpnet-base-v2

**Classifier:**
- Logistic Regression with balanced class weights

##Bag-of-Words / TF–IDF classification
**Feature extraction:**
- TF–IDF over character n-grams (char_wb, 3–5)

**Classifier:**
- Logistic Regression with balanced class weights

###Evaluation
For each experiment, I reported:
- Accuracy
- Precision / recall / F1 per class (classification report)
- Confusion matrix

For LinkedIn evaluation, macro-average recall was used alongside accuracy to better reflect performance across imbalanced labels.



---

###Models used for the LinkedIn prediction:

embedding model:

paraphrase-multilingual-mpnet-base-v2, train/test-split 50/50, Logistic Regression, confidence threshold ( < 0.50 -> "Other", < 0.60 -> "Professional")

bow model:

v2s (full) + TF-ID character n-grams (3, 5) + LR + confidence threshold (0.40) -> "Other", (0.50) -> "Professional"

#Results

###Embedding + LR (paraphrase-multilingual-mpnet-base-v2)

**Seniority**
- **Best accuracy:** **0.5743** — v2 capped **400** / split **70/30**
- **Best avg recall:** **0.6297** — v2 capped **400** / split **70/30**

**Department**
- **Best accuracy:** **0.6994** — v2 capped **500** / split **70/30**
- **Best avg recall:** **0.5857** — v2 capped **300** / split **90/10**

**Interpretation**
- Embedding models generalize well on extended data and achieve stronger department accuracy than before.
- Label capping improves average recall, especially for department.


###BoW TF–IDF char n-grams (char_wb 3–5) + LR

**Seniority**
- **Best accuracy:** **0.60** — v2 capped **500** / split **80/20**
- **Best avg recall:** **0.67** — v2 capped **500** / split **80/20**

**Department**
- **Best accuracy:** **0.63** — v2 capped **500** / split **80/20**
- **Best avg recall:** **0.51** — v2 capped **300** / split **90/10**

**Interpretation**
- Character n-grams capture seniority cues very reliably and outperform embeddings in seniority recall.
- Performance is stable across different split ratios.
- Label capping yields small but consistent recall improvements without harming accuracy.


##Best models
###Department

**Model**
- Embedding-based Logistic Regression  
- Embedding model: "paraphrase-multilingual-mpnet-base-v2"
- Training data: extended v2, capped at **500** examples per label  
- Train/test split: **70/30**  
- Classifier: Logistic Regression with balanced class weights  

Compared to Bag-of-Words models, embeddings capture semantic similarities between job titles that are important for department prediction.

---

###Seniority

**Model**
- Bag-of-Words TF–IDF over character n-grams (char_wb, 3–5) + Logistic Regression  
- Training data: extended v2, capped at **500** examples per label  
- Train/test split: **80/20**  
- Classifier: Logistic Regression with balanced class weights  

Character n-grams reliably capture lexical patterns and abbreviations that strongly correlate with seniority levels.


#Best models

##Department prediction

Creating extended v2

In [78]:
CAP = 500
SEED = 42

df_department = pd.read_csv("/content/department-v2.csv")
df_pred_bow   = pd.read_csv("/content/df_t_notannotated_pred_bow_both_threshold.csv")

# cut (randomly) so no label has > 500 rows
df_department = (df_department.groupby("label", group_keys=False)
                            .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                            .reset_index(drop=True))

# add predicted rows
df_department = pd.concat(
    [df_department, df_pred_bow[["position","department"]].rename(columns={"position":"text","department":"label"})],
    ignore_index=True
)

print("dept max count:", df_department["label"].value_counts().max())
print("df_department:", df_department.shape)


dept max count: 1242
sen  max count: 1118
df_department: (4483, 2)
df_seniority : (4295, 2)


/tmp/ipython-input-4068878874.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
/tmp/ipython-input-4068878874.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))


Model

In [82]:
df = df_department
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all,
    test_size=0.30,
    random_state=42,
    stratify=y_all
)

# --- embeddings ---
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)


# --- training LR classifier on embeddings ---
clf_dept_LR = LogisticRegression(class_weight="balanced")
clf_dept_LR.fit(X_train_emb, y_train)

# --- eval on v2 split ---

y_pred = clf_dept_LR.predict(X_eval_emb)
print("v2 eval accuracy:", round(accuracy_score(y_eval, y_pred), 4))
print(classification_report(y_eval, y_pred, digits=4, zero_division=0))

# --- LinkedIn eval (annotated) ---
final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

# --- embedding ---
X_final_eval_emb = model.encode(final_eval["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)

# --- predict ---
y_pred = clf_dept_LR.predict(X_final_eval_emb)
y_true = final_eval["department"].tolist()

# --- results ---
all_labels = sorted(set(clf_dept_LR.classes_) | set(y_true))

print("Logistic Regression accuracy:", round(np.mean(np.array(y_pred) == np.array(y_true)), 4))
print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
cm_df

Batches:   0%|          | 0/99 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

v2 eval accuracy: 0.8297
                        precision    recall  f1-score   support

        Administrative     0.5854    0.7742    0.6667        31
  Business Development     0.9231    0.9341    0.9286       167
            Consulting     0.8021    0.9390    0.8652        82
      Customer Support     0.5217    0.8571    0.6486        14
       Human Resources     0.7083    0.8947    0.7907        19
Information Technology     0.7525    0.7340    0.7431       203
             Marketing     0.9102    0.8994    0.9048       169
                 Other     0.8631    0.7265    0.7889       373
    Project Management     0.8144    0.9294    0.8681        85
            Purchasing     0.7407    1.0000    0.8511        20
                 Sales     0.8413    0.8736    0.8571       182

              accuracy                         0.8297      1345
             macro avg     0.7693    0.8693    0.8103      1345
          weighted avg     0.8361    0.8297    0.8295      1345



Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Logistic Regression accuracy: 0.6994
                        precision    recall  f1-score   support

        Administrative     0.3086    0.2976    0.3030        84
  Business Development     0.3925    0.5385    0.4541        78
            Consulting     0.7881    0.6103    0.6879       195
      Customer Support     0.6486    0.5000    0.5647        48
       Human Resources     0.7241    0.6000    0.6562        70
Information Technology     0.6324    0.5513    0.5890       312
             Marketing     0.6768    0.5038    0.5776       133
                 Other     0.7627    0.8267    0.7934      1252
    Project Management     0.7548    0.6686    0.7091       175
            Purchasing     0.4943    0.5972    0.5409        72
                 Sales     0.6795    0.7260    0.7020       219

              accuracy                         0.6994      2638
             macro avg     0.6239    0.5836    0.5980      2638
          weighted avg     0.7016    0.6994    0.6970      2638



,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,25,2,0,0,0,15,0,32,2,4,4
Business Development,0,42,0,0,0,0,1,30,1,0,4
Consulting,1,14,119,0,3,7,4,41,4,0,2
Customer Support,2,1,2,24,0,4,0,7,0,0,8
Human Resources,1,0,2,0,42,1,2,20,0,1,1
Information Technology,0,10,5,8,2,172,0,105,7,1,2
Marketing,3,2,6,3,1,17,67,30,0,2,2
Other,44,28,13,1,8,25,15,1035,20,26,37
Project Management,0,4,1,0,2,19,3,17,117,1,11
Purchasing,4,0,0,0,0,4,0,16,1,43,4


##Seniority Prediction

Creating extended v2

In [83]:
CAP = 500
SEED = 42

df_seniority  = pd.read_csv("/content/seniority-v2.csv")[["text","label"]]
df_pred_emb   = pd.read_csv("/content/df_t_notannotated_pred_embeddings_both_threshold.csv")

# cut (randomly) so no label has > 500 rows

df_seniority  = (df_seniority.groupby("label", group_keys=False)
                           .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                           .reset_index(drop=True))

# add predicted rows
df_seniority = pd.concat(
    [df_seniority, df_pred_emb[["position","seniority"]].rename(columns={"position":"text","seniority":"label"})],
    ignore_index=True
)

print("sen  max count:", df_seniority["label"].value_counts().max())
print("df_seniority :", df_seniority.shape)

dept max count: 1126
sen  max count: 1002
df_department: (4483, 2)
df_seniority : (4295, 2)


/tmp/ipython-input-1518447948.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
/tmp/ipython-input-1518447948.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))


Model

In [84]:
X_all = df_seniority["text"]
y_all = df_seniority["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all
)

X_test = df_t_annotated["position"]
y_test = df_t_annotated["seniority"]

clf_sen_char = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000)),
])

clf_sen_char.fit(X_train, y_train)

y_pred = clf_sen_char.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df


Accuracy: 0.5978013646702047
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.49      0.71      0.58       230
        Lead       0.95      0.55      0.70       460
  Management       0.91      0.63      0.74       418
Professional       0.70      0.53      0.60      1219
      Senior       0.19      0.72      0.30       170

    accuracy                           0.60      2638
   macro avg       0.66      0.67      0.62      2638
weighted avg       0.73      0.60      0.63      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,126,0,1,1,10,3
Junior,1,164,1,1,50,13
Lead,1,7,252,5,109,86
Management,43,2,5,262,79,27
Professional,0,150,6,19,650,394
Senior,2,12,0,1,32,123


#Loops to find best v2-shape and trainings/test-splits

##Embedding Loops

###embeddings on complete extended v2

In [85]:
df_department = pd.read_csv("/content/department-v2.csv")
df_seniority  = pd.read_csv("/content/seniority-v2.csv")
df_pred_bow   = pd.read_csv("/content/df_t_notannotated_pred_bow_both_threshold.csv")

df_department = pd.concat(
    [
        df_department[["text", "label"]],
        df_pred_bow[["position", "department"]].rename(
            columns={"position": "text", "department": "label"}
        )
    ],
    ignore_index=True
)

df_seniority = pd.concat(
    [
        df_seniority[["text", "label"]],
        df_pred_bow[["position", "seniority"]].rename(
            columns={"position": "text", "seniority": "label"}
        )
    ],
    ignore_index=True
)

print("df_department:", df_department.shape)
print("df_seniority :", df_seniority.shape)

df_department: (12031, 2)
df_seniority : (11314, 2)


####department-v2

In [86]:
df = df_department.copy()

model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

X_final_eval_emb = model.encode(
    final_eval["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)
y_true = final_eval["department"].tolist()


# LOOP OVER SPLITS


split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all,
        test_size=test_size,
        random_state=42,
        stratify=y_all
    )

    # --- embeddings (train + test) ---
    X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
    X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)

    # --- train LR ---
    clf_LR = LogisticRegression(class_weight="balanced")
    clf_LR.fit(X_train_emb, y_train)

    # --- v2-eval ---
    y_pred_eval = clf_LR.predict(X_eval_emb)
    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("v2 eval accuracy:", round(v2_acc, 4))


    # LinkedIn eval

    y_pred = clf_LR.predict(X_final_eval_emb)

    all_labels = sorted(set(clf_LR.classes_) | set(y_true))

    lk_acc = np.mean(np.array(y_pred) == np.array(y_true))
    lk_macro_recall = recall_score(
        y_true, y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    print("Logistic Regression accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
    display(cm_df)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)


Batches:   0%|          | 0/83 [00:00<?, ?it/s]


Split: test/train = 10/90


Batches:   0%|          | 0/339 [00:00<?, ?it/s]

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

v2 eval accuracy: 0.8912
Logistic Regression accuracy: 0.6857
LinkedIn macro recall: 0.5516
                        precision    recall  f1-score   support

        Administrative     0.3056    0.2619    0.2821        84
  Business Development     0.3846    0.5128    0.4396        78
            Consulting     0.7905    0.6000    0.6822       195
      Customer Support     0.8148    0.4583    0.5867        48
       Human Resources     0.8039    0.5857    0.6777        70
Information Technology     0.5891    0.3814    0.4630       312
             Marketing     0.7010    0.5113    0.5913       133
                 Other     0.7169    0.8618    0.7827      1252
    Project Management     0.7467    0.6400    0.6892       175
            Purchasing     0.6000    0.5833    0.5915        72
                 Sales     0.6934    0.6712    0.6821       219

              accuracy                         0.6857      2638
             macro avg     0.6497    0.5516    0.5880      2638
          

,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,22,2,0,0,0,11,0,39,0,4,6
Business Development,0,40,0,1,1,0,1,29,1,0,5
Consulting,2,15,117,0,3,7,4,42,3,0,2
Customer Support,3,2,2,22,0,1,0,14,0,0,4
Human Resources,0,0,3,0,41,1,1,23,0,1,0
Information Technology,0,11,5,3,2,119,0,160,8,1,3
Marketing,3,1,7,1,0,16,68,29,4,2,2
Other,37,22,10,0,4,23,16,1079,18,10,33
Project Management,0,5,1,0,0,15,2,34,112,1,5
Purchasing,3,0,0,0,0,3,0,18,1,42,5


... (22 outputs left out)

Batches:   0%|          | 0/226 [00:00<?, ?it/s]

v2 eval accuracy: 0.8802
Logistic Regression accuracy: 0.6748
LinkedIn macro recall: 0.537
                        precision    recall  f1-score   support

        Administrative     0.2985    0.2381    0.2649        84
  Business Development     0.3942    0.5256    0.4505        78
            Consulting     0.8041    0.6103    0.6939       195
      Customer Support     0.6296    0.3542    0.4533        48
       Human Resources     0.7455    0.5857    0.6560        70
Information Technology     0.6368    0.4103    0.4990       312
             Marketing     0.6827    0.5338    0.5992       133
                 Other     0.7059    0.8474    0.7702      1252
    Project Management     0.7468    0.6571    0.6991       175
            Purchasing     0.6119    0.5694    0.5899        72
                 Sales     0.6058    0.5753    0.5902       219

              accuracy                         0.6748      2638
             macro avg     0.6238    0.5370    0.5697      2638
          w

,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,20,2,0,1,1,10,0,40,1,3,6
Business Development,0,41,0,0,1,0,2,28,1,0,5
Consulting,3,14,119,0,3,8,5,39,2,0,2
Customer Support,2,2,4,17,0,4,0,13,0,0,6
Human Resources,1,0,2,0,41,1,1,23,0,1,0
Information Technology,0,9,4,6,2,128,0,149,11,1,2
Marketing,2,2,5,0,0,15,71,34,1,0,3
Other,36,25,13,0,6,16,14,1061,20,14,47
Project Management,0,5,1,0,1,11,2,34,115,1,5
Purchasing,2,0,0,0,0,3,0,20,0,41,6


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.891196,0.685747,0.551623
1,0.2,0.8,0.895721,0.683472,0.548333
2,0.3,0.7,0.889751,0.681577,0.545615
3,0.4,0.6,0.889466,0.683093,0.545393
4,0.5,0.5,0.884973,0.679682,0.540742
5,0.6,0.4,0.880177,0.674754,0.537031


####seniority-v2

In [87]:
df = df_seniority.copy()

model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


final_eval = df_t_annotated[["position", "seniority"]].copy()
final_eval["text"] = final_eval["position"]

X_final_eval_emb = model.encode(
    final_eval["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)
y_true = final_eval["seniority"].tolist()


# LOOP OVER SPLITS

split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all,
        test_size=test_size,
        random_state=42,
        stratify=y_all
    )

    # --- embeddings (train + test) ---
    X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
    X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)

    # --- train LR ---
    clf_LR = LogisticRegression(class_weight="balanced")
    clf_LR.fit(X_train_emb, y_train)

    # --- v2-eval ---
    y_pred_eval = clf_LR.predict(X_eval_emb)
    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("v2 eval accuracy:", round(v2_acc, 4))


    # LinkedIn eval

    y_pred = clf_LR.predict(X_final_eval_emb)

    all_labels = sorted(set(clf_LR.classes_) | set(y_true))

    lk_acc = np.mean(np.array(y_pred) == np.array(y_true))
    lk_macro_recall = recall_score(
        y_true, y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    print("Logistic Regression accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
    display(cm_df)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)


Batches:   0%|          | 0/83 [00:00<?, ?it/s]


Split: test/train = 10/90


Batches:   0%|          | 0/319 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

v2 eval accuracy: 0.8878
Logistic Regression accuracy: 0.5701
LinkedIn macro recall: 0.6093
              precision    recall  f1-score   support

    Director     0.6851    0.8794    0.7702       141
      Junior     0.3409    0.2609    0.2956       230
        Lead     0.8474    0.5674    0.6797       460
  Management     0.8427    0.6794    0.7523       418
Professional     0.6750    0.5332    0.5958      1219
      Senior     0.1857    0.7353    0.2966       170

    accuracy                         0.5701      2638
   macro avg     0.5961    0.6093    0.5650      2638
weighted avg     0.6715    0.5701    0.5991      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,124,2,2,4,6,3
Junior,1,60,4,2,152,11
Lead,5,12,261,24,74,84
Management,42,1,3,284,56,32
Professional,5,89,37,20,650,418
Senior,4,12,1,3,25,125


... (22 outputs left out)

Batches:   0%|          | 0/213 [00:00<?, ?it/s]

v2 eval accuracy: 0.8601
Logistic Regression accuracy: 0.5788
LinkedIn macro recall: 0.6107
              precision    recall  f1-score   support

    Director     0.6944    0.8865    0.7788       141
      Junior     0.3314    0.2522    0.2864       230
        Lead     0.8355    0.5630    0.6727       460
  Management     0.8519    0.7153    0.7776       418
Professional     0.6834    0.5472    0.6077      1219
      Senior     0.1842    0.7000    0.2917       170

    accuracy                         0.5788      2638
   macro avg     0.5968    0.6107    0.5692      2638
weighted avg     0.6743    0.5788    0.6068      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,125,1,2,4,6,3
Junior,1,58,4,2,153,12
Lead,5,14,259,24,77,81
Management,39,0,4,299,45,31
Professional,6,89,39,18,667,400
Senior,4,13,2,4,28,119


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.887809,0.570129,0.609273
1,0.2,0.8,0.878038,0.570508,0.612155
2,0.3,0.7,0.881591,0.570887,0.612081
3,0.4,0.6,0.873619,0.567854,0.610055
4,0.5,0.5,0.866183,0.574299,0.614046
5,0.6,0.4,0.860068,0.578848,0.610704


###embeddings on extended v2 with original v2 limited to 500 per label


In [77]:
CAP = 500
SEED = 42

df_department = pd.read_csv("/content/department-v2.csv")
df_seniority  = pd.read_csv("/content/seniority-v2.csv")
df_pred_bow   = pd.read_csv("/content/df_t_notannotated_pred_bow_both_threshold.csv")

# cut (randomly) so no label has > 500 rows
df_department = (df_department.groupby("label", group_keys=False)
                            .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                            .reset_index(drop=True))

df_seniority  = (df_seniority.groupby("label", group_keys=False)
                           .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                           .reset_index(drop=True))

# add predicted rows
df_department = pd.concat(
    [df_department, df_pred_bow[["position","department"]].rename(columns={"position":"text","department":"label"})],
    ignore_index=True
)

df_seniority = pd.concat(
    [df_seniority, df_pred_bow[["position","seniority"]].rename(columns={"position":"text","seniority":"label"})],
    ignore_index=True
)

print("dept max count:", df_department["label"].value_counts().max())
print("sen  max count:", df_seniority["label"].value_counts().max())


print("df_department:", df_department.shape)
print("df_seniority :", df_seniority.shape)

dept max count: 1242
sen  max count: 1118
df_department: (4483, 2)
df_seniority : (4295, 2)


/tmp/ipython-input-4068878874.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
/tmp/ipython-input-4068878874.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))


####department-v2

In [60]:
df = df_department.copy()

model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

X_final_eval_emb = model.encode(
    final_eval["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)
y_true = final_eval["department"].tolist()


# LOOP OVER SPLITS


split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all,
        test_size=test_size,
        random_state=42,
        stratify=y_all
    )

    # --- embeddings (train + test) ---
    X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
    X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)

    # --- train LR ---
    clf_LR = LogisticRegression(class_weight="balanced")
    clf_LR.fit(X_train_emb, y_train)

    # --- v2-eval ---
    y_pred_eval = clf_LR.predict(X_eval_emb)
    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("v2 eval accuracy:", round(v2_acc, 4))


    # LinkedIn eval

    y_pred = clf_LR.predict(X_final_eval_emb)

    all_labels = sorted(set(clf_LR.classes_) | set(y_true))

    lk_acc = np.mean(np.array(y_pred) == np.array(y_true))
    lk_macro_recall = recall_score(
        y_true, y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    print("Logistic Regression accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
    display(cm_df)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)


Batches:   0%|          | 0/83 [00:00<?, ?it/s]


Split: test/train = 10/90


Batches:   0%|          | 0/127 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

v2 eval accuracy: 0.8196
Logistic Regression accuracy: 0.6922
LinkedIn macro recall: 0.5738
                        precision    recall  f1-score   support

        Administrative     0.2889    0.3095    0.2989        84
  Business Development     0.3846    0.5128    0.4396        78
            Consulting     0.8027    0.6051    0.6901       195
      Customer Support     0.6571    0.4792    0.5542        48
       Human Resources     0.7241    0.6000    0.6562        70
Information Technology     0.6145    0.5160    0.5610       312
             Marketing     0.6947    0.4962    0.5789       133
                 Other     0.7527    0.8243    0.7869      1252
    Project Management     0.7452    0.6686    0.7048       175
            Purchasing     0.5190    0.5694    0.5430        72
                 Sales     0.6667    0.7306    0.6972       219

              accuracy                         0.6922      2638
             macro avg     0.6228    0.5738    0.5919      2638
          

,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,26,2,0,0,0,13,0,33,1,5,4
Business Development,0,40,0,1,0,0,1,30,1,0,5
Consulting,2,13,118,0,3,9,4,41,4,0,1
Customer Support,3,1,3,23,0,3,0,8,0,0,7
Human Resources,0,0,2,0,42,1,1,22,0,1,1
Information Technology,0,11,3,7,2,161,0,118,7,0,3
Marketing,2,3,6,3,1,16,66,29,3,2,2
Other,49,25,11,1,8,29,15,1032,20,20,42
Project Management,0,4,1,0,2,19,3,17,117,1,11
Purchasing,6,0,0,0,0,4,0,16,1,41,4


... (22 outputs left out)

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

v2 eval accuracy: 0.8238
Logistic Regression accuracy: 0.6888
LinkedIn macro recall: 0.574
                        precision    recall  f1-score   support

        Administrative     0.3288    0.2857    0.3057        84
  Business Development     0.3717    0.5385    0.4398        78
            Consulting     0.7707    0.6205    0.6875       195
      Customer Support     0.6000    0.5000    0.5455        48
       Human Resources     0.7241    0.6000    0.6562        70
Information Technology     0.5797    0.5128    0.5442       312
             Marketing     0.6364    0.4737    0.5431       133
                 Other     0.7649    0.8211    0.7920      1252
    Project Management     0.7188    0.6571    0.6866       175
            Purchasing     0.4943    0.5972    0.5409        72
                 Sales     0.6710    0.7078    0.6889       219

              accuracy                         0.6888      2638
             macro avg     0.6055    0.5740    0.5846      2638
          w

,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,24,2,0,1,0,15,0,32,2,4,4
Business Development,0,42,0,0,0,0,2,30,1,0,3
Consulting,2,14,121,0,3,8,4,38,3,2,0
Customer Support,2,1,3,24,0,5,0,5,0,0,8
Human Resources,1,0,3,0,42,1,2,19,0,1,1
Information Technology,0,11,6,9,2,160,0,113,8,1,2
Marketing,3,5,6,2,1,17,63,30,0,1,5
Other,37,29,14,1,9,30,15,1028,26,24,39
Project Management,0,5,1,1,1,24,5,13,115,0,10
Purchasing,3,0,0,0,0,6,0,14,2,43,4


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.819599,0.692191,0.573800
1,0.2,0.8,0.830546,0.692949,0.576418
2,0.3,0.7,0.829740,0.699393,0.583625
3,0.4,0.6,0.830546,0.699393,0.578749
4,0.5,0.5,0.831847,0.697877,0.577642
5,0.6,0.4,0.823792,0.688779,0.574037


####seniority-v2

In [61]:
df = df_seniority.copy()

model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


final_eval = df_t_annotated[["position", "seniority"]].copy()
final_eval["text"] = final_eval["position"]

X_final_eval_emb = model.encode(
    final_eval["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)
y_true = final_eval["seniority"].tolist()


# LOOP OVER SPLITS

split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all,
        test_size=test_size,
        random_state=42,
        stratify=y_all
    )

    # --- embeddings (train + test) ---
    X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
    X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)

    # --- train LR ---
    clf_LR = LogisticRegression(class_weight="balanced")
    clf_LR.fit(X_train_emb, y_train)

    # --- v2-eval ---
    y_pred_eval = clf_LR.predict(X_eval_emb)
    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("v2 eval accuracy:", round(v2_acc, 4))


    # LinkedIn eval

    y_pred = clf_LR.predict(X_final_eval_emb)

    all_labels = sorted(set(clf_LR.classes_) | set(y_true))

    lk_acc = np.mean(np.array(y_pred) == np.array(y_true))
    lk_macro_recall = recall_score(
        y_true, y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    print("Logistic Regression accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
    display(cm_df)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)


Batches:   0%|          | 0/83 [00:00<?, ?it/s]


Split: test/train = 10/90


Batches:   0%|          | 0/121 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

v2 eval accuracy: 0.8721
Logistic Regression accuracy: 0.5667
LinkedIn macro recall: 0.6235
              precision    recall  f1-score   support

    Director     0.6978    0.9007    0.7864       141
      Junior     0.3837    0.2870    0.3284       230
        Lead     0.7778    0.6239    0.6924       460
  Management     0.8665    0.6986    0.7735       418
Professional     0.6966    0.4897    0.5751      1219
      Senior     0.1748    0.7412    0.2828       170

    accuracy                         0.5667      2638
   macro avg     0.5995    0.6235    0.5731      2638
weighted avg     0.6768    0.5667    0.5980      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,127,1,3,3,5,2
Junior,1,66,5,3,135,20
Lead,4,13,287,19,53,84
Management,41,0,8,292,47,30
Professional,5,78,63,17,597,459
Senior,4,14,3,3,20,126


... (22 outputs left out)

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

v2 eval accuracy: 0.8223
Logistic Regression accuracy: 0.5591
LinkedIn macro recall: 0.6186
              precision    recall  f1-score   support

    Director     0.6906    0.8865    0.7764       141
      Junior     0.4228    0.2739    0.3325       230
        Lead     0.7696    0.6174    0.6852       460
  Management     0.8725    0.7201    0.7890       418
Professional     0.6841    0.4725    0.5590      1219
      Senior     0.1676    0.7412    0.2733       170

    accuracy                         0.5591      2638
   macro avg     0.6012    0.6186    0.5692      2638
weighted avg     0.6731    0.5591    0.5909      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,125,0,4,3,6,3
Junior,1,63,9,0,142,15
Lead,7,10,284,22,50,87
Management,39,0,5,301,42,31
Professional,5,67,65,16,576,490
Senior,4,9,2,3,26,126


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.872093,0.566717,0.623511
1,0.2,0.8,0.855646,0.567854,0.620999
2,0.3,0.7,0.854151,0.564064,0.621105
3,0.4,0.6,0.846915,0.557619,0.616311
4,0.5,0.5,0.830540,0.559136,0.617226
5,0.6,0.4,0.822274,0.559136,0.618603


###embeddings on extended v2 with original v2 limited to 400 per label

In [62]:
CAP = 400
SEED = 42

df_department = pd.read_csv("/content/department-v2.csv")
df_seniority  = pd.read_csv("/content/seniority-v2.csv")
df_pred_bow   = pd.read_csv("/content/df_t_notannotated_pred_bow_both_threshold.csv")

# cut (randomly) so no label has > 500 rows
df_department = (df_department.groupby("label", group_keys=False)
                            .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                            .reset_index(drop=True))

df_seniority  = (df_seniority.groupby("label", group_keys=False)
                           .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                           .reset_index(drop=True))

# add predicted rows
df_department = pd.concat(
    [df_department, df_pred_bow[["position","department"]].rename(columns={"position":"text","department":"label"})],
    ignore_index=True
)

df_seniority = pd.concat(
    [df_seniority, df_pred_bow[["position","seniority"]].rename(columns={"position":"text","seniority":"label"})],
    ignore_index=True
)

print("dept max count:", df_department["label"].value_counts().max())
print("sen  max count:", df_seniority["label"].value_counts().max())


print("df_department:", df_department.shape)
print("df_seniority :", df_seniority.shape)

dept max count: 1242
sen  max count: 1018
df_department: (4083, 2)
df_seniority : (3886, 2)


/tmp/ipython-input-1278291270.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
/tmp/ipython-input-1278291270.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))


####department-v2

In [63]:
df = df_department.copy()

model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


final_eval = df_t_annotated[["position", "department"]].copy()
final_eval["text"] = final_eval["position"]

X_final_eval_emb = model.encode(
    final_eval["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)
y_true = final_eval["department"].tolist()


# LOOP OVER SPLITS


split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all,
        test_size=test_size,
        random_state=42,
        stratify=y_all
    )

    # --- embeddings (train + test) ---
    X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
    X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)

    # --- train LR ---
    clf_LR = LogisticRegression(class_weight="balanced")
    clf_LR.fit(X_train_emb, y_train)

    # --- v2-eval ---
    y_pred_eval = clf_LR.predict(X_eval_emb)
    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("v2 eval accuracy:", round(v2_acc, 4))


    # LinkedIn eval

    y_pred = clf_LR.predict(X_final_eval_emb)

    all_labels = sorted(set(clf_LR.classes_) | set(y_true))

    lk_acc = np.mean(np.array(y_pred) == np.array(y_true))
    lk_macro_recall = recall_score(
        y_true, y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    print("Logistic Regression accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
    display(cm_df)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)


Batches:   0%|          | 0/83 [00:00<?, ?it/s]


Split: test/train = 10/90


Batches:   0%|          | 0/115 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

v2 eval accuracy: 0.8289
Logistic Regression accuracy: 0.6945
LinkedIn macro recall: 0.578
                        precision    recall  f1-score   support

        Administrative     0.2935    0.3214    0.3068        84
  Business Development     0.3761    0.5256    0.4385        78
            Consulting     0.7881    0.6103    0.6879       195
      Customer Support     0.6111    0.4583    0.5238        48
       Human Resources     0.7119    0.6000    0.6512        70
Information Technology     0.6130    0.5737    0.5927       312
             Marketing     0.6875    0.4962    0.5764       133
                 Other     0.7696    0.8163    0.7922      1252
    Project Management     0.7170    0.6514    0.6826       175
            Purchasing     0.5185    0.5833    0.5490        72
                 Sales     0.6723    0.7215    0.6960       219

              accuracy                         0.6945      2638
             macro avg     0.6144    0.5780    0.5907      2638
          w

,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,27,2,0,0,0,14,0,30,2,5,4
Business Development,0,41,0,1,1,0,1,29,1,0,4
Consulting,2,14,119,0,3,9,4,40,3,0,1
Customer Support,3,1,3,22,0,4,0,6,0,0,9
Human Resources,1,0,2,0,42,1,1,21,0,1,1
Information Technology,0,11,4,7,2,179,0,98,7,1,3
Marketing,3,3,6,3,1,17,66,27,3,2,2
Other,50,26,13,3,8,33,16,1022,25,20,36
Project Management,0,5,1,0,2,20,3,18,114,1,11
Purchasing,4,0,0,0,0,5,0,14,1,42,6


... (22 outputs left out)

Batches:   0%|          | 0/77 [00:00<?, ?it/s]

v2 eval accuracy: 0.8114
Logistic Regression accuracy: 0.6865
LinkedIn macro recall: 0.5812
                        precision    recall  f1-score   support

        Administrative     0.3118    0.3452    0.3277        84
  Business Development     0.3894    0.5641    0.4607        78
            Consulting     0.7973    0.6051    0.6880       195
      Customer Support     0.5682    0.5208    0.5435        48
       Human Resources     0.7288    0.6143    0.6667        70
Information Technology     0.6103    0.5321    0.5685       312
             Marketing     0.6667    0.4662    0.5487       133
                 Other     0.7672    0.8107    0.7883      1252
    Project Management     0.7616    0.6571    0.7055       175
            Purchasing     0.4941    0.5833    0.5350        72
                 Sales     0.5914    0.6941    0.6387       219

              accuracy                         0.6865      2638
             macro avg     0.6079    0.5812    0.5883      2638
          

,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
Administrative,29,2,0,0,0,12,0,31,2,4,4
Business Development,0,44,0,0,1,0,1,29,0,0,3
Consulting,3,13,118,1,5,8,5,36,4,0,2
Customer Support,3,1,3,25,0,3,0,4,0,0,9
Human Resources,2,0,2,0,43,1,1,19,0,1,1
Information Technology,1,12,2,11,1,166,0,108,7,1,3
Marketing,3,4,5,4,2,18,62,27,2,2,4
Other,47,26,14,2,6,31,15,1015,18,22,56
Project Management,0,5,1,0,1,19,2,15,115,1,16
Purchasing,3,0,0,1,0,3,0,16,0,42,7


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.828851,0.694466,0.578012
1,0.2,0.8,0.829865,0.690296,0.570905
2,0.3,0.7,0.808163,0.691433,0.574831
3,0.4,0.6,0.802938,0.690296,0.572272
4,0.5,0.5,0.806072,0.680819,0.564645
5,0.6,0.4,0.811429,0.686505,0.581186


####seniority-v2

In [64]:
df = df_seniority.copy()

model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


final_eval = df_t_annotated[["position", "seniority"]].copy()
final_eval["text"] = final_eval["position"]

X_final_eval_emb = model.encode(
    final_eval["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)
y_true = final_eval["seniority"].tolist()


# LOOP OVER SPLITS

split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all,
        test_size=test_size,
        random_state=42,
        stratify=y_all
    )

    # --- embeddings (train + test) ---
    X_train_emb = model.encode(X_train.tolist(), normalize_embeddings=True, show_progress_bar=True)
    X_eval_emb  = model.encode(X_eval.tolist(),  normalize_embeddings=True, show_progress_bar=True)

    # --- train LR ---
    clf_LR = LogisticRegression(class_weight="balanced")
    clf_LR.fit(X_train_emb, y_train)

    # --- v2-eval ---
    y_pred_eval = clf_LR.predict(X_eval_emb)
    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("v2 eval accuracy:", round(v2_acc, 4))


    # LinkedIn eval

    y_pred = clf_LR.predict(X_final_eval_emb)

    all_labels = sorted(set(clf_LR.classes_) | set(y_true))

    lk_acc = np.mean(np.array(y_pred) == np.array(y_true))
    lk_macro_recall = recall_score(
        y_true, y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    print("Logistic Regression accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_true, y_pred, labels=all_labels, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
    display(cm_df)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)


Batches:   0%|          | 0/83 [00:00<?, ?it/s]


Split: test/train = 10/90


Batches:   0%|          | 0/110 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

v2 eval accuracy: 0.856
Logistic Regression accuracy: 0.5697
LinkedIn macro recall: 0.6253
              precision    recall  f1-score   support

    Director     0.6923    0.8936    0.7802       141
      Junior     0.3567    0.2652    0.3042       230
        Lead     0.7838    0.6304    0.6988       460
  Management     0.8699    0.7201    0.7880       418
Professional     0.7048    0.4897    0.5779      1219
      Senior     0.1773    0.7529    0.2870       170

    accuracy                         0.5697      2638
   macro avg     0.5975    0.6253    0.5727      2638
weighted avg     0.6797    0.5697    0.6005      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,126,2,3,3,5,2
Junior,1,61,5,3,141,19
Lead,4,15,290,18,50,83
Management,42,0,10,301,35,30
Professional,5,80,59,18,597,460
Senior,4,13,3,3,19,128


... (22 outputs left out)

Batches:   0%|          | 0/73 [00:00<?, ?it/s]

v2 eval accuracy: 0.8229
Logistic Regression accuracy: 0.569
LinkedIn macro recall: 0.6204
              precision    recall  f1-score   support

    Director     0.6971    0.8652    0.7722       141
      Junior     0.3964    0.2913    0.3358       230
        Lead     0.8088    0.5978    0.6875       460
  Management     0.8543    0.7297    0.7871       418
Professional     0.6982    0.4971    0.5807      1219
      Senior     0.1728    0.7412    0.2803       170

    accuracy                         0.5690      2638
   macro avg     0.6046    0.6204    0.5739      2638
weighted avg     0.6820    0.5690    0.6016      2638



,Director,Junior,Lead,Management,Professional,Senior
Director,122,0,3,5,9,2
Junior,1,67,4,3,144,11
Lead,5,14,275,26,52,88
Management,40,0,8,305,35,30
Professional,3,75,48,15,606,472
Senior,4,13,2,3,22,126


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.856041,0.569750,0.625342
1,0.2,0.8,0.841902,0.570887,0.623156
2,0.3,0.7,0.835334,0.574299,0.629687
3,0.4,0.6,0.834727,0.570129,0.625130
4,0.5,0.5,0.829645,0.572782,0.626943
5,0.6,0.4,0.822899,0.568992,0.620391


##Bag of Words Loops

###bow on complete extended v2

In [88]:
df_department = pd.read_csv("/content/department-v2.csv")
df_seniority  = pd.read_csv("/content/seniority-v2.csv")
df_pred_emb   = pd.read_csv("/content/df_t_notannotated_pred_embeddings_both_threshold.csv")
df_department = pd.concat(
    [
        df_department[["text", "label"]],
        df_pred_emb[["position", "department"]].rename(
            columns={"position": "text", "department": "label"}
        )
    ],
    ignore_index=True
)

df_seniority = pd.concat(
    [
        df_seniority[["text", "label"]],
        df_pred_emb[["position", "seniority"]].rename(
            columns={"position": "text", "seniority": "label"}
        )
    ],
    ignore_index=True
)

print("df_department:", df_department.shape)
print("df_seniority :", df_seniority.shape)

df_department: (12031, 2)
df_seniority : (11314, 2)


####department-v2

In [89]:
df = df_department.copy()


final_eval = df_t_annotated[["position", "department"]].copy()
X_test = final_eval["position"]
y_test = final_eval["department"].to_numpy()


# LOOP OVER SPLITS

split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all, test_size=test_size, random_state=42, stratify=y_all
    )

    clf_dept_char_split = Pipeline([
        ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
        ("tfidf", TfidfTransformer()),
        ("clf", LogisticRegression(class_weight="balanced")),
    ])

    clf_dept_char_split.fit(X_train, y_train)

    # --- eval on department-v2-test-split ---
    y_pred_eval = clf_dept_char_split.predict(X_eval)

    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("test-split Accuracy:", round(v2_acc, 4))


    # --- eval on LinkedIn annotated ---
    y_pred_li = clf_dept_char_split.predict(X_test)

    lk_acc = accuracy_score(y_test, y_pred_li)
    labels = sorted(set(y_eval) | set(y_pred_eval))
    lk_macro_recall = recall_score(y_test, y_pred_li, labels=labels, average="macro", zero_division=0)

    print("LinkedIn Accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_test, y_pred_li, zero_division=0))

    labels_li = sorted(set(y_test) | set(y_pred_li))
    cm = confusion_matrix(y_test, y_pred_li, labels=labels_li)
    cm_df_2 = pd.DataFrame(
        cm,
        index=pd.Index(labels_li, name="true"),
        columns=pd.Index(labels_li, name="pred"),
    )
    display(cm_df_2)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)



Split: test/train = 10/90
test-split Accuracy: 0.9269
LinkedIn Accuracy: 0.6232
LinkedIn macro recall: 0.5009
                        precision    recall  f1-score   support

        Administrative       0.17      0.20      0.18        84
  Business Development       0.34      0.44      0.38        78
            Consulting       0.51      0.71      0.59       195
      Customer Support       0.72      0.48      0.57        48
       Human Resources       0.74      0.60      0.66        70
Information Technology       0.55      0.21      0.31       312
             Marketing       0.71      0.38      0.50       133
                 Other       0.64      0.79      0.71      1252
    Project Management       0.80      0.66      0.72       175
            Purchasing       0.67      0.43      0.53        72
                 Sales       0.90      0.60      0.72       219

              accuracy                           0.62      2638
             macro avg       0.61      0.50      0.53  

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,1,9,0,49,0,6,0
Business Development,0,34,15,0,1,3,1,21,0,0,3
Consulting,0,17,139,0,3,1,3,28,3,0,1
Customer Support,3,1,1,23,0,3,0,17,0,0,0
Human Resources,1,0,7,0,42,3,0,16,0,0,1
Information Technology,2,7,13,6,0,66,0,203,13,1,1
Marketing,3,1,3,1,0,11,51,62,0,0,1
Other,69,32,92,2,9,16,14,994,10,7,7
Project Management,0,4,1,0,1,5,0,48,115,0,1



Split: test/train = 20/80
test-split Accuracy: 0.9215
LinkedIn Accuracy: 0.6209
LinkedIn macro recall: 0.4929
                        precision    recall  f1-score   support

        Administrative       0.16      0.19      0.17        84
  Business Development       0.36      0.45      0.40        78
            Consulting       0.52      0.71      0.60       195
      Customer Support       0.72      0.44      0.55        48
       Human Resources       0.74      0.60      0.66        70
Information Technology       0.59      0.21      0.31       312
             Marketing       0.70      0.38      0.50       133
                 Other       0.63      0.80      0.71      1252
    Project Management       0.80      0.66      0.72       175
            Purchasing       0.67      0.43      0.53        72
                 Sales       0.90      0.55      0.68       219

              accuracy                           0.62      2638
             macro avg       0.62      0.49      0.53  

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,16,2,0,0,1,8,0,51,0,6,0
Business Development,0,35,15,0,1,3,1,21,0,0,2
Consulting,0,16,138,0,3,1,3,29,4,0,1
Customer Support,3,1,1,21,0,3,0,19,0,0,0
Human Resources,1,0,7,0,42,1,0,18,0,0,1
Information Technology,2,7,12,5,0,67,0,205,12,1,1
Marketing,3,1,3,1,0,11,51,62,0,0,1
Other,71,31,85,2,9,14,14,1001,11,7,7
Project Management,0,4,1,0,1,4,0,49,115,0,1



Split: test/train = 30/70
test-split Accuracy: 0.9172
LinkedIn Accuracy: 0.6194
LinkedIn macro recall: 0.4927
                        precision    recall  f1-score   support

        Administrative       0.18      0.21      0.20        84
  Business Development       0.37      0.45      0.40        78
            Consulting       0.51      0.70      0.59       195
      Customer Support       0.65      0.46      0.54        48
       Human Resources       0.72      0.59      0.65        70
Information Technology       0.59      0.23      0.33       312
             Marketing       0.70      0.38      0.50       133
                 Other       0.64      0.80      0.71      1252
    Project Management       0.78      0.65      0.71       175
            Purchasing       0.69      0.43      0.53        72
                 Sales       0.90      0.52      0.66       219

              accuracy                           0.62      2638
             macro avg       0.61      0.49      0.53  

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,18,2,0,0,1,8,0,50,0,5,0
Business Development,0,35,15,0,1,2,1,21,1,0,2
Consulting,0,14,137,0,4,1,3,30,5,0,1
Customer Support,3,1,1,22,0,3,0,18,0,0,0
Human Resources,2,0,7,0,41,1,0,18,0,0,1
Information Technology,2,7,12,6,0,72,0,199,12,1,1
Marketing,3,1,3,1,0,12,51,61,0,0,1
Other,67,30,89,5,9,15,14,1000,11,6,6
Project Management,0,4,1,0,1,7,0,48,113,0,1


... (3 outputs left out)

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,19,2,0,0,0,8,0,49,0,6,0
Business Development,0,35,14,0,1,2,1,22,1,0,2
Consulting,0,14,139,0,3,1,3,30,4,0,1
Customer Support,3,1,1,22,0,4,0,17,0,0,0
Human Resources,2,0,6,0,38,2,0,20,0,0,2
Information Technology,1,7,11,7,1,68,0,204,8,4,1
Marketing,3,1,4,1,1,12,50,60,0,0,1
Other,69,30,84,5,8,17,14,1003,8,7,7
Project Management,0,4,1,0,1,3,0,52,113,0,1



Split: test/train = 60/40
test-split Accuracy: 0.9041
LinkedIn Accuracy: 0.6194
LinkedIn macro recall: 0.4835
                        precision    recall  f1-score   support

        Administrative       0.17      0.21      0.19        84
  Business Development       0.37      0.45      0.40        78
            Consulting       0.54      0.71      0.62       195
      Customer Support       0.64      0.44      0.52        48
       Human Resources       0.86      0.51      0.64        70
Information Technology       0.53      0.23      0.32       312
             Marketing       0.68      0.38      0.49       133
                 Other       0.64      0.80      0.71      1252
    Project Management       0.83      0.65      0.73       175
            Purchasing       0.61      0.42      0.50        72
                 Sales       0.88      0.51      0.65       219

              accuracy                           0.62      2638
             macro avg       0.61      0.48      0.52  

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,18,2,0,0,0,8,0,50,0,6,0
Business Development,0,35,14,0,1,2,1,22,1,0,2
Consulting,0,14,139,0,3,3,3,28,4,0,1
Customer Support,3,1,1,21,0,4,0,18,0,0,0
Human Resources,2,0,6,0,36,6,0,19,0,0,1
Information Technology,1,7,10,7,0,72,0,205,5,4,1
Marketing,3,1,4,0,0,13,50,61,0,0,1
Other,71,30,78,5,1,20,14,1007,11,7,8
Project Management,0,3,1,0,1,5,0,50,114,0,1


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.926910,0.623199,0.500876
1,0.2,0.8,0.921479,0.620925,0.492938
2,0.3,0.7,0.917175,0.619409,0.492671
3,0.4,0.6,0.916268,0.619788,0.497044
4,0.5,0.5,0.913896,0.617134,0.485803
5,0.6,0.4,0.904142,0.619409,0.483468


####seniority-v2

In [90]:
df = df_seniority.copy()


final_eval = df_t_annotated[["position", "seniority"]].copy()
X_test = final_eval["position"]
y_test = final_eval["seniority"].to_numpy()


# LOOP OVER SPLITS

split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all, test_size=test_size, random_state=42, stratify=y_all
    )

    clf_sen_char = Pipeline([
        ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
        ("tfidf", TfidfTransformer()),
        ("clf", LogisticRegression(class_weight="balanced")),
    ])

    clf_sen_char.fit(X_train, y_train)

    # --- eval on seniority-v2-test-split ---
    y_pred_eval = clf_sen_char.predict(X_eval)

    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("test-split Accuracy:", round(v2_acc, 4))

    # --- eval on LinkedIn annotated ---
    y_pred_li = clf_sen_char.predict(X_test)

    lk_acc = accuracy_score(y_test, y_pred_li)
    labels_li = sorted(set(y_test) | set(y_pred_li))
    lk_macro_recall = recall_score(y_test, y_pred_li, labels=labels_li, average="macro", zero_division=0)

    print("LinkedIn Accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_test, y_pred_li, zero_division=0))

    cm = confusion_matrix(y_test, y_pred_li, labels=labels_li)
    cm_df = pd.DataFrame(
        cm,
        index=pd.Index(labels_li, name="true"),
        columns=pd.Index(labels_li, name="pred"),
    )
    display(cm_df)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)



Split: test/train = 10/90
test-split Accuracy: 0.9435
LinkedIn Accuracy: 0.5879
LinkedIn macro recall: 0.6572
              precision    recall  f1-score   support

    Director       0.74      0.89      0.81       141
      Junior       0.45      0.70      0.55       230
        Lead       0.94      0.49      0.65       460
  Management       0.90      0.64      0.75       418
Professional       0.68      0.54      0.60      1219
      Senior       0.19      0.68      0.30       170

    accuracy                           0.59      2638
   macro avg       0.65      0.66      0.61      2638
weighted avg       0.71      0.59      0.62      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,126,0,1,1,10,3
Junior,1,161,1,1,57,9
Lead,0,6,227,6,130,91
Management,42,2,9,266,71,28
Professional,0,174,3,21,655,366
Senior,2,12,0,1,39,116



Split: test/train = 20/80
test-split Accuracy: 0.9364
LinkedIn Accuracy: 0.5902
LinkedIn macro recall: 0.6586
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.46      0.70      0.55       230
        Lead       0.93      0.49      0.64       460
  Management       0.90      0.65      0.75       418
Professional       0.69      0.54      0.61      1219
      Senior       0.19      0.69      0.30       170

    accuracy                           0.59      2638
   macro avg       0.65      0.66      0.61      2638
weighted avg       0.71      0.59      0.62      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,0,1,1,11,3
Junior,1,161,1,1,55,11
Lead,0,8,226,7,127,92
Management,43,2,9,270,67,27
Professional,0,169,5,20,658,367
Senior,2,12,0,1,38,117



Split: test/train = 30/70
test-split Accuracy: 0.9272
LinkedIn Accuracy: 0.5842
LinkedIn macro recall: 0.6578
              precision    recall  f1-score   support

    Director       0.74      0.89      0.80       141
      Junior       0.44      0.70      0.54       230
        Lead       0.93      0.50      0.65       460
  Management       0.90      0.63      0.74       418
Professional       0.68      0.53      0.59      1219
      Senior       0.19      0.71      0.30       170

    accuracy                           0.58      2638
   macro avg       0.65      0.66      0.61      2638
weighted avg       0.71      0.58      0.61      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,0,0,1,12,3
Junior,1,161,1,1,54,12
Lead,0,7,229,7,127,90
Management,42,1,9,263,77,26
Professional,0,181,6,18,643,371
Senior,2,12,0,1,35,120


... (3 outputs left out)

pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,127,0,0,1,10,3
Junior,1,162,2,1,54,10
Lead,1,7,231,5,126,90
Management,42,1,9,260,80,26
Professional,0,163,8,21,646,381
Senior,2,12,0,1,47,108



Split: test/train = 60/40
test-split Accuracy: 0.9121
LinkedIn Accuracy: 0.572
LinkedIn macro recall: 0.6346
              precision    recall  f1-score   support

    Director       0.73      0.88      0.79       141
      Junior       0.45      0.67      0.54       230
        Lead       0.91      0.52      0.66       460
  Management       0.89      0.63      0.73       418
Professional       0.66      0.52      0.58      1219
      Senior       0.17      0.60      0.26       170

    accuracy                           0.57      2638
   macro avg       0.63      0.63      0.59      2638
weighted avg       0.69      0.57      0.61      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,124,0,1,1,12,3
Junior,1,154,2,1,62,10
Lead,1,5,237,6,120,91
Management,42,1,9,262,78,26
Professional,0,168,11,25,630,385
Senior,3,12,0,1,52,102


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.943463,0.587945,0.657190
1,0.2,0.8,0.936368,0.590220,0.658631
2,0.3,0.7,0.927246,0.584155,0.657817
3,0.4,0.6,0.923774,0.584534,0.652859
4,0.5,0.5,0.916740,0.581501,0.649080
5,0.6,0.4,0.912064,0.572024,0.634638


In [91]:
X_train = df_seniority["text"]
y_train = df_seniority["label"]

X_test = df_t_annotated["position"]
y_test = df_t_annotated["seniority"]

clf_sen_char = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3,5),)),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced",)),
    ])
clf_sen_char.fit(X_train, y_train)

y_pred = clf_sen_char.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.5921152388172858
              precision    recall  f1-score   support

    Director       0.74      0.89      0.80       141
      Junior       0.47      0.71      0.56       230
        Lead       0.94      0.49      0.65       460
  Management       0.90      0.63      0.74       418
Professional       0.69      0.54      0.61      1219
      Senior       0.19      0.71      0.31       170

    accuracy                           0.59      2638
   macro avg       0.65      0.66      0.61      2638
weighted avg       0.72      0.59      0.62      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,0,1,1,11,3
Junior,1,164,1,1,53,10
Lead,0,7,227,6,129,91
Management,42,2,9,262,77,26
Professional,0,166,3,20,664,366
Senior,2,12,0,1,35,120


###bow on extended v2 with original v2 limited to 500 per label

In [69]:
CAP = 500
SEED = 42

df_department = pd.read_csv("/content/department-v2.csv")[["text","label"]]
df_seniority  = pd.read_csv("/content/seniority-v2.csv")[["text","label"]]
df_pred_emb   = pd.read_csv("/content/df_t_notannotated_pred_embeddings_both_threshold.csv")

# cut (randomly) so no label has > 500 rows
df_department = (df_department.groupby("label", group_keys=False)
                            .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                            .reset_index(drop=True))

df_seniority  = (df_seniority.groupby("label", group_keys=False)
                           .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                           .reset_index(drop=True))

# add predicted rows
df_department = pd.concat(
    [df_department, df_pred_emb[["position","department"]].rename(columns={"position":"text","department":"label"})],
    ignore_index=True
)

df_seniority = pd.concat(
    [df_seniority, df_pred_emb[["position","seniority"]].rename(columns={"position":"text","seniority":"label"})],
    ignore_index=True
)

print("dept max count:", df_department["label"].value_counts().max())
print("sen  max count:", df_seniority["label"].value_counts().max())


print("df_department:", df_department.shape)
print("df_seniority :", df_seniority.shape)

dept max count: 1126
sen  max count: 1002
df_department: (4483, 2)
df_seniority : (4295, 2)


/tmp/ipython-input-1518447948.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
/tmp/ipython-input-1518447948.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))


####department-v2

In [70]:
df = df_department.copy()


final_eval = df_t_annotated[["position", "department"]].copy()
X_test = final_eval["position"]
y_test = final_eval["department"].to_numpy()


# LOOP OVER SPLITS

split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all, test_size=test_size, random_state=42, stratify=y_all
    )

    clf_dept_char_split = Pipeline([
        ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
        ("tfidf", TfidfTransformer()),
        ("clf", LogisticRegression(class_weight="balanced")),
    ])

    clf_dept_char_split.fit(X_train, y_train)

    # --- eval on department-v2-test-split ---
    y_pred_eval = clf_dept_char_split.predict(X_eval)

    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("test-split Accuracy:", round(v2_acc, 4))

    # --- eval on LinkedIn annotated ---
    y_pred_li = clf_dept_char_split.predict(X_test)

    lk_acc = accuracy_score(y_test, y_pred_li)
    labels = sorted(set(y_eval) | set(y_pred_eval))
    lk_macro_recall = recall_score(y_test, y_pred_li, labels=labels, average="macro", zero_division=0)

    print("LinkedIn Accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_test, y_pred_li, zero_division=0))

    labels_li = sorted(set(y_test) | set(y_pred_li))
    cm = confusion_matrix(y_test, y_pred_li, labels=labels_li)
    cm_df_2 = pd.DataFrame(
        cm,
        index=pd.Index(labels_li, name="true"),
        columns=pd.Index(labels_li, name="pred"),
    )
    display(cm_df_2)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)



Split: test/train = 10/90
test-split Accuracy: 0.8797
LinkedIn Accuracy: 0.6243
LinkedIn macro recall: 0.5003
                        precision    recall  f1-score   support

        Administrative       0.16      0.20      0.18        84
  Business Development       0.34      0.44      0.38        78
            Consulting       0.55      0.69      0.62       195
      Customer Support       0.75      0.50      0.60        48
       Human Resources       0.75      0.61      0.68        70
Information Technology       0.48      0.25      0.33       312
             Marketing       0.69      0.38      0.49       133
                 Other       0.65      0.80      0.72      1252
    Project Management       0.84      0.64      0.73       175
            Purchasing       0.65      0.42      0.51        72
                 Sales       0.87      0.57      0.69       219

              accuracy                           0.62      2638
             macro avg       0.61      0.50      0.54  

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,0,15,0,44,0,6,0
Business Development,0,34,15,0,1,3,1,21,0,0,3
Consulting,0,17,135,0,3,3,4,29,3,0,1
Customer Support,3,1,1,24,0,5,0,14,0,0,0
Human Resources,2,0,6,0,43,2,0,15,0,0,2
Information Technology,2,8,10,5,2,78,0,199,6,1,1
Marketing,3,1,3,1,0,15,51,58,0,0,1
Other,76,31,72,2,7,27,14,998,9,7,9
Project Management,0,4,1,0,1,13,0,43,112,0,1



Split: test/train = 20/80
test-split Accuracy: 0.8662
LinkedIn Accuracy: 0.6293
LinkedIn macro recall: 0.5033
                        precision    recall  f1-score   support

        Administrative       0.16      0.20      0.18        84
  Business Development       0.36      0.45      0.40        78
            Consulting       0.56      0.69      0.61       195
      Customer Support       0.77      0.48      0.59        48
       Human Resources       0.79      0.60      0.68        70
Information Technology       0.46      0.26      0.33       312
             Marketing       0.69      0.39      0.50       133
                 Other       0.65      0.80      0.72      1252
    Project Management       0.85      0.65      0.73       175
            Purchasing       0.74      0.44      0.56        72
                 Sales       0.88      0.58      0.70       219

              accuracy                           0.63      2638
             macro avg       0.63      0.50      0.55  

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,0,16,0,48,0,1,0
Business Development,0,35,16,0,1,3,1,20,0,0,2
Consulting,0,16,134,0,3,3,4,31,3,0,1
Customer Support,3,1,1,23,0,5,0,15,0,0,0
Human Resources,2,0,6,0,42,4,0,14,0,0,2
Information Technology,2,7,9,5,1,81,0,198,5,3,1
Marketing,3,1,3,0,0,13,52,60,0,0,1
Other,73,29,70,2,5,32,14,1005,8,5,9
Project Management,0,4,1,0,1,14,0,41,113,0,1



Split: test/train = 30/70
test-split Accuracy: 0.858
LinkedIn Accuracy: 0.6285
LinkedIn macro recall: 0.5001
                        precision    recall  f1-score   support

        Administrative       0.16      0.20      0.18        84
  Business Development       0.36      0.45      0.40        78
            Consulting       0.59      0.68      0.63       195
      Customer Support       0.73      0.46      0.56        48
       Human Resources       0.75      0.63      0.68        70
Information Technology       0.45      0.26      0.33       312
             Marketing       0.69      0.39      0.50       133
                 Other       0.65      0.81      0.72      1252
    Project Management       0.86      0.65      0.74       175
            Purchasing       0.79      0.42      0.55        72
                 Sales       0.87      0.57      0.69       219

              accuracy                           0.63      2638
             macro avg       0.63      0.50      0.54   

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,0,15,0,50,0,0,0
Business Development,0,35,16,0,1,3,1,20,0,0,2
Consulting,0,17,133,0,3,3,4,31,3,0,1
Customer Support,3,1,1,22,0,5,0,16,0,0,0
Human Resources,2,0,6,0,44,2,0,14,0,0,2
Information Technology,2,8,7,5,0,80,0,200,6,3,1
Marketing,3,1,3,0,0,13,52,60,0,0,1
Other,72,29,59,3,10,35,14,1008,7,5,10
Project Management,0,4,1,0,1,14,0,41,113,0,1


... (3 outputs left out)

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,0,17,0,48,0,0,0
Business Development,0,36,13,0,1,4,1,22,0,0,1
Consulting,0,14,123,0,3,4,4,43,3,0,1
Customer Support,3,1,1,17,0,5,0,21,0,0,0
Human Resources,2,0,3,0,42,3,0,18,0,0,2
Information Technology,2,8,8,3,1,78,0,203,5,3,1
Marketing,3,1,3,0,0,13,50,62,0,0,1
Other,70,29,41,2,6,36,14,1028,7,6,13
Project Management,0,4,0,1,1,15,0,40,113,0,1



Split: test/train = 60/40
test-split Accuracy: 0.842
LinkedIn Accuracy: 0.6342
LinkedIn macro recall: 0.4824
                        precision    recall  f1-score   support

        Administrative       0.17      0.20      0.18        84
  Business Development       0.37      0.46      0.41        78
            Consulting       0.66      0.62      0.64       195
      Customer Support       0.74      0.35      0.48        48
       Human Resources       0.78      0.60      0.68        70
Information Technology       0.42      0.26      0.32       312
             Marketing       0.68      0.38      0.48       133
                 Other       0.65      0.83      0.73      1252
    Project Management       0.85      0.64      0.73       175
            Purchasing       0.89      0.35      0.50        72
                 Sales       0.85      0.62      0.72       219

              accuracy                           0.63      2638
             macro avg       0.64      0.48      0.53   

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,0,17,0,48,0,0,0
Business Development,0,36,12,0,1,3,1,23,1,0,1
Consulting,0,14,121,0,3,4,4,45,3,0,1
Customer Support,3,1,2,17,0,7,0,17,0,0,1
Human Resources,2,0,2,0,42,4,0,18,0,0,2
Information Technology,2,8,10,3,0,81,0,201,5,1,1
Marketing,3,1,3,0,0,13,50,62,0,0,1
Other,70,30,29,2,7,39,14,1037,7,2,15
Project Management,0,4,0,1,1,18,0,38,112,0,1


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.879733,0.624337,0.500263
1,0.2,0.8,0.866221,0.629265,0.503296
2,0.3,0.7,0.857993,0.628506,0.500104
3,0.4,0.6,0.855072,0.631539,0.501260
4,0.5,0.5,0.851918,0.630781,0.486534
5,0.6,0.4,0.842007,0.634193,0.482372


####seniority characters

In [71]:
df = df_seniority.copy()


final_eval = df_t_annotated[["position", "seniority"]].copy()
X_test = final_eval["position"]
y_test = final_eval["seniority"].to_numpy()


# LOOP OVER SPLITS

split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all, test_size=test_size, random_state=42, stratify=y_all
    )

    clf_sen_char = Pipeline([
        ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
        ("tfidf", TfidfTransformer()),
        ("clf", LogisticRegression(class_weight="balanced")),
    ])

    clf_sen_char.fit(X_train, y_train)

    # --- eval on seniority-v2-test-split ---
    y_pred_eval = clf_sen_char.predict(X_eval)

    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("test-split Accuracy:", round(v2_acc, 4))

    # --- eval on LinkedIn annotated ---
    y_pred_li = clf_sen_char.predict(X_test)

    lk_acc = accuracy_score(y_test, y_pred_li)
    labels_li = sorted(set(y_test) | set(y_pred_li))
    lk_macro_recall = recall_score(y_test, y_pred_li, labels=labels_li, average="macro", zero_division=0)

    print("LinkedIn Accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_test, y_pred_li, zero_division=0))

    cm = confusion_matrix(y_test, y_pred_li, labels=labels_li)
    cm_df = pd.DataFrame(
        cm,
        index=pd.Index(labels_li, name="true"),
        columns=pd.Index(labels_li, name="pred"),
    )
    display(cm_df)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)



Split: test/train = 10/90
test-split Accuracy: 0.8442
LinkedIn Accuracy: 0.5944
LinkedIn macro recall: 0.6678
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.48      0.70      0.57       230
        Lead       0.95      0.55      0.70       460
  Management       0.90      0.63      0.74       418
Professional       0.70      0.53      0.60      1219
      Senior       0.19      0.71      0.30       170

    accuracy                           0.59      2638
   macro avg       0.66      0.67      0.62      2638
weighted avg       0.72      0.59      0.63      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,0,1,1,11,3
Junior,1,161,2,1,49,16
Lead,1,5,253,5,109,87
Management,43,2,5,263,78,27
Professional,0,155,5,21,645,393
Senior,2,12,0,1,34,121



Split: test/train = 20/80
test-split Accuracy: 0.8673
LinkedIn Accuracy: 0.5978
LinkedIn macro recall: 0.673
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.49      0.71      0.58       230
        Lead       0.95      0.55      0.70       460
  Management       0.91      0.63      0.74       418
Professional       0.70      0.53      0.60      1219
      Senior       0.19      0.72      0.30       170

    accuracy                           0.60      2638
   macro avg       0.66      0.67      0.62      2638
weighted avg       0.73      0.60      0.63      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,126,0,1,1,10,3
Junior,1,164,1,1,50,13
Lead,1,7,252,5,109,86
Management,43,2,5,262,79,27
Professional,0,150,6,19,650,394
Senior,2,12,0,1,32,123



Split: test/train = 30/70
test-split Accuracy: 0.8635
LinkedIn Accuracy: 0.5967
LinkedIn macro recall: 0.672
              precision    recall  f1-score   support

    Director       0.73      0.90      0.81       141
      Junior       0.48      0.71      0.57       230
        Lead       0.94      0.55      0.69       460
  Management       0.91      0.63      0.74       418
Professional       0.70      0.53      0.60      1219
      Senior       0.19      0.72      0.30       170

    accuracy                           0.60      2638
   macro avg       0.66      0.67      0.62      2638
weighted avg       0.73      0.60      0.63      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,127,0,1,1,9,3
Junior,1,163,2,1,50,13
Lead,1,7,251,5,109,87
Management,42,2,5,262,79,28
Professional,0,156,7,17,649,390
Senior,2,13,0,1,32,122


... (3 outputs left out)

pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,128,0,1,1,8,3
Junior,1,146,2,1,64,16
Lead,1,7,248,7,113,84
Management,43,2,5,263,77,28
Professional,0,159,7,20,654,379
Senior,2,14,0,1,31,122



Split: test/train = 60/40
test-split Accuracy: 0.8487
LinkedIn Accuracy: 0.5872
LinkedIn macro recall: 0.6513
              precision    recall  f1-score   support

    Director       0.73      0.89      0.81       141
      Junior       0.44      0.62      0.51       230
        Lead       0.94      0.54      0.68       460
  Management       0.87      0.61      0.72       418
Professional       0.68      0.54      0.60      1219
      Senior       0.20      0.71      0.31       170

    accuracy                           0.59      2638
   macro avg       0.64      0.65      0.60      2638
weighted avg       0.71      0.59      0.62      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,126,0,2,1,9,3
Junior,1,142,2,2,71,12
Lead,0,7,247,11,114,81
Management,43,4,5,254,85,27
Professional,0,159,8,22,659,371
Senior,2,14,0,1,32,121


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.844186,0.594390,0.667766
1,0.2,0.8,0.867288,0.597801,0.673006
2,0.3,0.7,0.863460,0.596664,0.671984
3,0.4,0.6,0.862631,0.587945,0.656893
4,0.5,0.5,0.854283,0.591736,0.660842
5,0.6,0.4,0.848661,0.587187,0.651332


In [72]:
X_train = df_seniority["text"]
y_train = df_seniority["label"]

X_test = df_t_annotated["position"]
y_test = df_t_annotated["seniority"]

clf_sen_char = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3,5),)),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced",)),
    ])
clf_sen_char.fit(X_train, y_train)

y_pred = clf_sen_char.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.5898407884761183
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.47      0.67      0.55       230
        Lead       0.95      0.54      0.69       460
  Management       0.89      0.62      0.73       418
Professional       0.69      0.53      0.60      1219
      Senior       0.18      0.69      0.29       170

    accuracy                           0.59      2638
   macro avg       0.65      0.66      0.61      2638
weighted avg       0.72      0.59      0.62      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,0,1,1,11,3
Junior,1,155,3,1,58,12
Lead,1,6,250,6,109,88
Management,43,2,5,260,81,27
Professional,0,155,4,22,648,390
Senior,2,12,0,1,37,118


###bow on extended v2 with original v2 limited to 300 per label

In [73]:
CAP = 300
SEED = 42

df_department = pd.read_csv("/content/department-v2.csv")[["text","label"]]
df_seniority  = pd.read_csv("/content/seniority-v2.csv")[["text","label"]]
df_pred_emb   = pd.read_csv("/content/df_t_notannotated_pred_embeddings_both_threshold.csv")

# cut (randomly) so no label has > 500 rows
df_department = (df_department.groupby("label", group_keys=False)
                            .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                            .reset_index(drop=True))

df_seniority  = (df_seniority.groupby("label", group_keys=False)
                           .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
                           .reset_index(drop=True))

# add predicted rows
df_department = pd.concat(
    [df_department, df_pred_emb[["position","department"]].rename(columns={"position":"text","department":"label"})],
    ignore_index=True
)

df_seniority = pd.concat(
    [df_seniority, df_pred_emb[["position","seniority"]].rename(columns={"position":"text","seniority":"label"})],
    ignore_index=True
)

print("dept max count:", df_department["label"].value_counts().max())
print("sen  max count:", df_seniority["label"].value_counts().max())


print("df_department:", df_department.shape)
print("df_seniority :", df_seniority.shape)

dept max count: 1126
sen  max count: 802
df_department: (3683, 2)
df_seniority : (3386, 2)


/tmp/ipython-input-2530380094.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))
/tmp/ipython-input-2530380094.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), CAP), random_state=SEED))


####department-v2

In [74]:
df = df_department.copy()


final_eval = df_t_annotated[["position", "department"]].copy()
X_test = final_eval["position"]
y_test = final_eval["department"].to_numpy()


# LOOP OVER SPLITS

split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all, test_size=test_size, random_state=42, stratify=y_all
    )

    clf_dept_char_split = Pipeline([
        ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
        ("tfidf", TfidfTransformer()),
        ("clf", LogisticRegression(class_weight="balanced")),
    ])

    clf_dept_char_split.fit(X_train, y_train)

    # --- eval on department-v2-test-split ---
    y_pred_eval = clf_dept_char_split.predict(X_eval)

    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("test-split Accuracy:", round(v2_acc, 4))

    # --- eval on LinkedIn annotated ---
    y_pred_li = clf_dept_char_split.predict(X_test)

    lk_acc = accuracy_score(y_test, y_pred_li)
    labels = sorted(set(y_eval) | set(y_pred_eval))
    lk_macro_recall = recall_score(y_test, y_pred_li, labels=labels, average="macro", zero_division=0)

    print("LinkedIn Accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_test, y_pred_li, zero_division=0))

    labels_li = sorted(set(y_test) | set(y_pred_li))
    cm = confusion_matrix(y_test, y_pred_li, labels=labels_li)
    cm_df_2 = pd.DataFrame(
        cm,
        index=pd.Index(labels_li, name="true"),
        columns=pd.Index(labels_li, name="pred"),
    )
    display(cm_df_2)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)



Split: test/train = 10/90
test-split Accuracy: 0.8428
LinkedIn Accuracy: 0.6198
LinkedIn macro recall: 0.5058
                        precision    recall  f1-score   support

        Administrative       0.16      0.20      0.18        84
  Business Development       0.34      0.45      0.39        78
            Consulting       0.57      0.70      0.63       195
      Customer Support       0.71      0.60      0.65        48
       Human Resources       0.75      0.63      0.68        70
Information Technology       0.46      0.26      0.33       312
             Marketing       0.69      0.40      0.50       133
                 Other       0.64      0.79      0.71      1252
    Project Management       0.86      0.65      0.74       175
            Purchasing       0.66      0.40      0.50        72
                 Sales       0.87      0.47      0.62       219

              accuracy                           0.62      2638
             macro avg       0.61      0.51      0.54  

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,1,16,0,42,0,6,0
Business Development,0,35,14,0,1,3,1,22,0,0,2
Consulting,0,16,137,0,4,3,4,27,3,0,1
Customer Support,3,1,1,29,0,3,0,11,0,0,0
Human Resources,2,0,7,0,44,4,0,12,0,0,1
Information Technology,2,8,7,8,2,82,0,195,5,2,1
Marketing,3,1,3,0,0,14,53,58,0,0,1
Other,73,33,69,4,5,40,15,992,7,6,8
Project Management,0,4,1,0,1,10,0,45,113,0,1



Split: test/train = 20/80
test-split Accuracy: 0.8331
LinkedIn Accuracy: 0.616
LinkedIn macro recall: 0.5083
                        precision    recall  f1-score   support

        Administrative       0.16      0.20      0.18        84
  Business Development       0.35      0.45      0.39        78
            Consulting       0.57      0.72      0.63       195
      Customer Support       0.69      0.65      0.67        48
       Human Resources       0.74      0.64      0.69        70
Information Technology       0.44      0.25      0.32       312
             Marketing       0.68      0.38      0.49       133
                 Other       0.64      0.79      0.71      1252
    Project Management       0.85      0.65      0.73       175
            Purchasing       0.66      0.40      0.50        72
                 Sales       0.87      0.47      0.61       219

              accuracy                           0.62      2638
             macro avg       0.60      0.51      0.54   

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,1,15,0,43,0,6,0
Business Development,0,35,16,0,1,3,1,20,0,0,2
Consulting,0,17,140,0,3,3,4,24,3,0,1
Customer Support,3,1,1,31,0,4,0,8,0,0,0
Human Resources,2,0,7,0,45,3,0,12,0,0,1
Information Technology,2,8,7,8,2,78,0,199,5,2,1
Marketing,3,1,3,0,0,15,51,59,0,0,1
Other,74,31,70,6,6,44,15,984,8,6,8
Project Management,0,4,1,0,2,10,0,44,113,0,1



Split: test/train = 30/70
test-split Accuracy: 0.8362
LinkedIn Accuracy: 0.6122
LinkedIn macro recall: 0.5081
                        precision    recall  f1-score   support

        Administrative       0.17      0.20      0.18        84
  Business Development       0.37      0.47      0.42        78
            Consulting       0.56      0.71      0.63       195
      Customer Support       0.70      0.67      0.68        48
       Human Resources       0.67      0.64      0.66        70
Information Technology       0.41      0.24      0.30       312
             Marketing       0.69      0.41      0.51       133
                 Other       0.64      0.78      0.70      1252
    Project Management       0.83      0.65      0.73       175
            Purchasing       0.60      0.35      0.44        72
                 Sales       0.89      0.47      0.61       219

              accuracy                           0.61      2638
             macro avg       0.59      0.51      0.53  

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,1,18,0,40,0,6,0
Business Development,0,37,15,0,1,3,1,20,0,0,1
Consulting,0,15,139,0,4,4,4,24,4,0,1
Customer Support,2,1,1,32,0,3,0,9,0,0,0
Human Resources,2,0,7,0,45,3,0,12,0,0,1
Information Technology,2,8,7,8,3,75,0,201,6,1,1
Marketing,3,1,3,0,0,15,54,56,0,0,1
Other,72,30,72,6,11,46,15,975,9,8,8
Project Management,0,4,1,0,1,10,0,45,114,0,0


... (3 outputs left out)

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,1,0,16,0,45,0,3,0
Business Development,0,37,14,0,1,3,1,21,0,0,1
Consulting,0,18,132,0,3,5,4,29,3,0,1
Customer Support,3,0,1,31,0,3,0,8,0,0,2
Human Resources,2,0,7,0,42,3,0,15,0,0,1
Information Technology,1,8,5,8,1,76,0,207,5,0,1
Marketing,3,1,3,0,0,13,52,60,0,0,1
Other,72,30,65,5,10,58,15,976,7,6,8
Project Management,0,4,1,0,1,7,0,49,113,0,0



Split: test/train = 60/40
test-split Accuracy: 0.8312
LinkedIn Accuracy: 0.6039
LinkedIn macro recall: 0.4786
                        precision    recall  f1-score   support

        Administrative       0.17      0.20      0.18        84
  Business Development       0.36      0.46      0.40        78
            Consulting       0.55      0.67      0.61       195
      Customer Support       0.69      0.46      0.55        48
       Human Resources       0.71      0.56      0.62        70
Information Technology       0.40      0.28      0.33       312
             Marketing       0.64      0.39      0.49       133
                 Other       0.63      0.77      0.70      1252
    Project Management       0.85      0.65      0.73       175
            Purchasing       0.70      0.36      0.48        72
                 Sales       0.87      0.47      0.61       219

              accuracy                           0.60      2638
             macro avg       0.60      0.48      0.52  

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,17,2,0,0,0,14,0,46,0,5,0
Business Development,0,36,14,1,1,4,1,20,0,0,1
Consulting,0,17,131,0,3,5,4,31,3,0,1
Customer Support,4,0,1,22,0,7,0,14,0,0,0
Human Resources,2,0,8,0,39,8,0,12,0,0,1
Information Technology,1,8,5,6,1,86,0,199,5,0,1
Marketing,3,1,3,0,1,14,52,58,0,0,1
Other,71,30,70,3,8,61,19,969,8,4,9
Project Management,0,4,1,0,1,6,0,49,113,0,1


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.842818,0.619788,0.505766
1,0.2,0.8,0.833107,0.615997,0.508308
2,0.3,0.7,0.836199,0.612206,0.508058
3,0.4,0.6,0.828358,0.613723,0.504686
4,0.5,0.5,0.823561,0.608415,0.498313
5,0.6,0.4,0.831222,0.603867,0.478577


####seniority-v2

In [75]:
df = df_seniority.copy()


final_eval = df_t_annotated[["position", "seniority"]].copy()
X_test = final_eval["position"]
y_test = final_eval["seniority"].to_numpy()


# LOOP OVER SPLITS

split_results = []

for test_size in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]:
    print("\n" + "=" * 70)
    print(f"Split: test/train = {int(test_size*100)}/{int((1-test_size)*100)}")

    # --- split ---
    X_all = df["text"]
    y_all = df["label"]

    X_train, X_eval, y_train, y_eval = train_test_split(
        X_all, y_all, test_size=test_size, random_state=42, stratify=y_all
    )

    clf_sen_char = Pipeline([
        ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
        ("tfidf", TfidfTransformer()),
        ("clf", LogisticRegression(class_weight="balanced")),
    ])

    clf_sen_char.fit(X_train, y_train)

    # --- eval on seniority-v2-test-split ---
    y_pred_eval = clf_sen_char.predict(X_eval)

    v2_acc = accuracy_score(y_eval, y_pred_eval)
    print("test-split Accuracy:", round(v2_acc, 4))

    # --- eval on LinkedIn annotated ---
    y_pred_li = clf_sen_char.predict(X_test)

    lk_acc = accuracy_score(y_test, y_pred_li)
    labels_li = sorted(set(y_test) | set(y_pred_li))
    lk_macro_recall = recall_score(y_test, y_pred_li, labels=labels_li, average="macro",zero_division=0)

    print("LinkedIn Accuracy:", round(lk_acc, 4))
    print("LinkedIn macro recall:", round(lk_macro_recall, 4))
    print(classification_report(y_test, y_pred_li, zero_division=0))

    cm = confusion_matrix(y_test, y_pred_li, labels=labels_li)
    cm_df = pd.DataFrame(
        cm,
        index=pd.Index(labels_li, name="true"),
        columns=pd.Index(labels_li, name="pred"),
    )
    display(cm_df)

    split_results.append({
        "test_size": test_size,
        "train_size": 1 - test_size,
        "v2_eval_acc": v2_acc,
        "linkedin_acc": lk_acc,
        "linkedin_macro_recall": lk_macro_recall
    })

results_df = pd.DataFrame(split_results).sort_values("test_size")
display(results_df)



Split: test/train = 10/90
test-split Accuracy: 0.8407
LinkedIn Accuracy: 0.5929
LinkedIn macro recall: 0.6686
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.46      0.70      0.56       230
        Lead       0.95      0.55      0.70       460
  Management       0.88      0.64      0.74       418
Professional       0.70      0.52      0.60      1219
      Senior       0.19      0.71      0.30       170

    accuracy                           0.59      2638
   macro avg       0.65      0.67      0.62      2638
weighted avg       0.72      0.59      0.63      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,126,0,2,1,9,3
Junior,1,162,1,1,51,14
Lead,1,5,252,10,107,85
Management,43,2,5,266,74,28
Professional,0,167,5,23,638,386
Senior,2,14,0,1,33,120



Split: test/train = 20/80
test-split Accuracy: 0.8555
LinkedIn Accuracy: 0.5914
LinkedIn macro recall: 0.6672
              precision    recall  f1-score   support

    Director       0.73      0.89      0.81       141
      Junior       0.46      0.69      0.55       230
        Lead       0.94      0.56      0.70       460
  Management       0.88      0.63      0.73       418
Professional       0.69      0.52      0.59      1219
      Senior       0.19      0.71      0.30       170

    accuracy                           0.59      2638
   macro avg       0.65      0.67      0.61      2638
weighted avg       0.72      0.59      0.62      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,126,0,2,1,10,2
Junior,1,159,1,1,53,15
Lead,0,5,256,10,107,82
Management,43,2,5,263,78,27
Professional,0,170,7,22,635,385
Senior,2,13,0,1,33,121



Split: test/train = 30/70
test-split Accuracy: 0.8474
LinkedIn Accuracy: 0.5895
LinkedIn macro recall: 0.6647
              precision    recall  f1-score   support

    Director       0.73      0.90      0.81       141
      Junior       0.46      0.68      0.55       230
        Lead       0.95      0.55      0.69       460
  Management       0.87      0.63      0.73       418
Professional       0.69      0.52      0.59      1219
      Senior       0.19      0.71      0.30       170

    accuracy                           0.59      2638
   macro avg       0.65      0.66      0.61      2638
weighted avg       0.71      0.59      0.62      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,127,0,1,1,11,1
Junior,1,156,1,3,54,15
Lead,1,5,251,11,113,79
Management,43,2,5,263,78,27
Professional,0,165,6,24,637,387
Senior,2,12,0,1,34,121


... (3 outputs left out)

pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,124,0,2,1,13,1
Junior,1,151,1,2,56,19
Lead,0,6,255,12,105,82
Management,43,4,5,253,85,28
Professional,1,159,11,17,661,370
Senior,2,15,0,1,43,109



Split: test/train = 60/40
test-split Accuracy: 0.8233
LinkedIn Accuracy: 0.5868
LinkedIn macro recall: 0.656
              precision    recall  f1-score   support

    Director       0.72      0.87      0.79       141
      Junior       0.45      0.68      0.54       230
        Lead       0.93      0.56      0.70       460
  Management       0.87      0.58      0.70       418
Professional       0.68      0.53      0.60      1219
      Senior       0.20      0.72      0.31       170

    accuracy                           0.59      2638
   macro avg       0.64      0.66      0.61      2638
weighted avg       0.70      0.59      0.62      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,123,0,2,1,14,1
Junior,1,156,1,1,53,18
Lead,1,5,256,9,118,71
Management,43,4,6,242,95,28
Professional,0,169,11,23,649,367
Senior,2,13,0,1,32,122


,test_size,train_size,v2_eval_acc,linkedin_acc,linkedin_macro_recall
0,0.1,0.9,0.840708,0.592873,0.668569
1,0.2,0.8,0.855457,0.591357,0.667219
2,0.3,0.7,0.847441,0.589462,0.664689
3,0.4,0.6,0.842804,0.591736,0.662643
4,0.5,0.5,0.847017,0.588704,0.646498
5,0.6,0.4,0.823327,0.586808,0.656020


In [76]:
X_train = df_seniority["text"]
y_train = df_seniority["label"]

X_test = df_t_annotated["position"]
y_test = df_t_annotated["seniority"]

clf_sen_char = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3,5),)),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced",)),
    ])
clf_sen_char.fit(X_train, y_train)

y_pred = clf_sen_char.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.590978013646702
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.47      0.70      0.56       230
        Lead       0.94      0.55      0.69       460
  Management       0.88      0.63      0.74       418
Professional       0.70      0.52      0.60      1219
      Senior       0.19      0.70      0.29       170

    accuracy                           0.59      2638
   macro avg       0.65      0.67      0.61      2638
weighted avg       0.72      0.59      0.62      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,0,2,1,10,3
Junior,1,162,2,1,51,13
Lead,1,6,252,10,102,89
Management,43,2,5,264,76,28
Professional,0,164,6,23,637,389
Senior,2,13,0,1,35,119
